# Fase 3 - Modelo clasico: Random Forest

En esta fase se entrena un modelo clasico de Machine Learning para el problema de clasificacion binaria del dataset Adult / Census Income.

El objetivo es predecir si una persona gana `<=50K` o `>50K` al anio. La variable objetivo ya esta codificada como:

- `0`: <=50K
- `1`: >50K

Se utiliza Random Forest porque es un modelo clasico adecuado para datos tabulares y permite comparar sus resultados contra los modelos de redes neuronales de la Fase 2.

## 1. Importar librerias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GridSearchCV

sns.set(style="whitegrid")

## 2. Cargar los datos procesados

Se usan directamente los archivos generados en la Fase 1. No se repite el preprocesamiento ni se modifica ningun CSV.

In [ ]:
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

print("Dimensiones de los datos:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

## 3. Funciones de evaluacion

Estas funciones permiten evaluar de la misma forma el Random Forest base y el Random Forest optimizado.

In [ ]:
def graficar_matriz_confusion(y_real, y_pred, titulo):
    matriz = confusion_matrix(y_real, y_pred)
    
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        matriz,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["<=50K", ">50K"],
        yticklabels=["<=50K", ">50K"],
    )
    plt.title(titulo)
    plt.xlabel("Prediccion")
    plt.ylabel("Valor real")
    plt.show()


def evaluar_modelo(nombre_modelo, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision_clase_1 = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    recall_clase_1 = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1_clase_1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    
    print(f"Resultados de {nombre_modelo}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision >50K: {precision_clase_1:.4f}")
    print(f"Recall >50K:    {recall_clase_1:.4f}")
    print(f"F1 >50K:        {f1_clase_1:.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"], zero_division=0))
    
    graficar_matriz_confusion(y_test, y_pred, f"Matriz de confusion - {nombre_modelo}")
    
    metricas = {
        "Modelo": nombre_modelo,
        "Accuracy": accuracy,
        "Precision >50K": precision_clase_1,
        "Recall >50K": recall_clase_1,
        "F1 >50K": f1_clase_1,
        "F1 macro": f1_macro,
        "F1 weighted": f1_weighted,
    }
    
    return metricas

## 4. Entrenar Random Forest base

Se entrena un primer Random Forest con parametros base. Se usa `class_weight="balanced"` porque la clase `>50K` suele ser minoritaria en este dataset.

In [ ]:
rf_base = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)

rf_base.fit(X_train, y_train)

## 5. Evaluar Random Forest base

In [ ]:
metricas_rf_base = evaluar_modelo(
    "Random Forest base",
    rf_base,
    X_test,
    y_test,
)

## 6. Ajuste basico de hiperparametros

Se usa una busqueda sencilla con pocos valores para mantener el proceso entendible y sustentable. La metrica de seleccion es `f1`, enfocada en la clase positiva `>50K`.

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "criterion": ["gini", "entropy"],
}

rf_grid = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)

grid_search = GridSearchCV(
    estimator=rf_grid,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Mejores hiperparametros encontrados:")
print(grid_search.best_params_)
print(f"Mejor F1 promedio en validacion cruzada: {grid_search.best_score_:.4f}")

## 7. Entrenar y evaluar el mejor Random Forest

`GridSearchCV` ya deja entrenado el mejor modelo usando los mejores parametros encontrados.

In [ ]:
mejor_rf = grid_search.best_estimator_

metricas_rf_optimizado = evaluar_modelo(
    "Random Forest optimizado",
    mejor_rf,
    X_test,
    y_test,
)

## 8. Tabla de metricas del Random Forest optimizado

In [ ]:
tabla_rf = pd.DataFrame([metricas_rf_optimizado])
tabla_rf

## 9. Comparacion con los modelos de Fase 2

Primero se intenta cargar automaticamente `metricas_fase2.csv`, generado al ejecutar el notebook de Fase 2.

Si ese archivo todavia no existe, se crea una tabla editable para escribir las metricas manualmente.

En ese caso, reemplaza los valores `np.nan` por las metricas reales de:

- Perceptron
- Red neuronal 1 capa oculta
- Red neuronal 2 capas ocultas

In [ ]:
try:
    metricas_fase2 = pd.read_csv("metricas_fase2.csv")
    print("Metricas de Fase 2 cargadas desde metricas_fase2.csv")
except FileNotFoundError:
    print("No se encontro metricas_fase2.csv. Completa manualmente esta tabla con los resultados de Fase 2.")
    metricas_fase2 = pd.DataFrame([
        {
            "Modelo": "Perceptron",
            "Accuracy": np.nan,
            "Precision >50K": np.nan,
            "Recall >50K": np.nan,
            "F1 >50K": np.nan,
            "F1 macro": np.nan,
            "F1 weighted": np.nan,
        },
        {
            "Modelo": "Red neuronal 1 capa oculta",
            "Accuracy": np.nan,
            "Precision >50K": np.nan,
            "Recall >50K": np.nan,
            "F1 >50K": np.nan,
            "F1 macro": np.nan,
            "F1 weighted": np.nan,
        },
        {
            "Modelo": "Red neuronal 2 capas ocultas",
            "Accuracy": np.nan,
            "Precision >50K": np.nan,
            "Recall >50K": np.nan,
            "F1 >50K": np.nan,
            "F1 macro": np.nan,
            "F1 weighted": np.nan,
        },
    ])

metricas_fase2

## 10. Tabla comparativa final

Se unen las metricas de Fase 2 con las del Random Forest optimizado. Luego se ordenan por `F1 >50K`, que es importante porque la clase `>50K` es la clase positiva y suele estar menos representada.

In [ ]:
tabla_comparativa = pd.concat([metricas_fase2, tabla_rf], ignore_index=True)

tabla_comparativa_ordenada = tabla_comparativa.sort_values(
    by="F1 >50K",
    ascending=False,
    na_position="last",
).reset_index(drop=True)

tabla_comparativa_ordenada

## 11. Conclusion automatica

La conclusion toma el mejor modelo segun `F1 >50K`. Si todavia faltan las metricas de Fase 2, la comparacion se basa en los valores disponibles.

In [ ]:
metricas_disponibles = tabla_comparativa.dropna(subset=["F1 >50K"])

if metricas_disponibles.empty:
    print("Todavia no hay metricas disponibles para comparar modelos.")
else:
    mejor_modelo = metricas_disponibles.sort_values("F1 >50K", ascending=False).iloc[0]
    print(
        "Segun la metrica F1 para la clase >50K, "
        f"el mejor modelo es {mejor_modelo['Modelo']} "
        f"con un F1 >50K de {mejor_modelo['F1 >50K']:.4f}."
    )